# Historical Transport Resilience Index (TRI) — Proof of Concept

**Type:** Historical / retrospective geospatial resilience assessment
**Scope:** This notebook computes the TRI from a single historical flood event using
observed-style (here: synthetically generated) road, flood, facility, and travel-survey data.

## What this notebook does

| Stage | Output |
|---|---|
| 1. Infrastructure Functionality (IF) | Road serviceability under a historical flood, by ward |
| 2. Accessibility Index (AI) | Share of essential facilities reachable through the surviving network, by ward |
| 3. Behavioural Adaptation (BA) | How travellers actually responded (delay, mode change, cancellations, rerouting), by ward |
| 4. Transport Resilience Index (TRI) | `TRI = (IF + AI + BA) / 3`, by ward |

## Explicitly out of scope

This is a **historical assessment tool only**. It does **not** contain, and will never contain:
- Machine Learning / Deep Learning
- Predictive or forecasting models
- Future / "what-if" scenario simulation
- LLMs, RAG, or any generative-AI component

Every number produced here describes **what already happened**, computed with transparent,
auditable, rule-based formulas.

## How the notebook is organised

The notebook is split into four parts (this is **Part 1**):

- **Part 1** — Project setup, package installation, folder structure, dummy data generation, helper utilities *(this notebook)*
- **Part 2** — Road–Flood Overlay → Road Status → Infrastructure Functionality (IF) → Accessibility Index (AI)
- **Part 3** — Travel Behaviour → Behavioural Adaptation (BA) → Master Dataset → Transport Resilience Index (TRI)
- **Part 4** — Visualization, validation reporting, and `run_pipeline()` end-to-end execution

Every intermediate result is written to disk as a shapefile / GeoTIFF / CSV, so each function in
later parts can be re-run independently by reading the files produced by the previous step.


## System Architecture

```
Road Network  ─┐
               ├──► Road–Flood Overlay ──► Road Status ──► Infrastructure Functionality (IF)
Flood Raster  ─┘                                                        │
                                                                         ▼
Essential Facilities ───────────────────────────────────────►  Accessibility Index (AI)
                                                                         │
Travel Survey ──► Travel Behaviour Metrics ──► Behavioural Adaptation (BA)
                                                                         │
                                                                         ▼
                                              IF + AI + BA ──► Master Dataset ──► TRI
```

**Reading order:** Road geometry is overlaid on the flood raster to get an average flood depth
per road segment → each segment is classified Operational / Partial / Closed → the *effective
length* of the network (weighted by that classification) per ward gives **IF**. The road-status
network (with Closed roads removed) is used as a graph to test which facilities each ward can
still reach, giving **AI**. Independently, the travel survey tells us what travellers actually
experienced (delay, cancellations, mode/route changes), aggregated per ward into **BA**. The
three ward-level indices are combined into the final **TRI**.


## Project folder structure

Everything is created automatically under `/content/project` (Colab) — nothing needs to be
uploaded manually.

```
project/
├── input/
│   ├── roads/          road_network.shp
│   ├── flood/           flood_depth.tif
│   ├── facilities/      facilities.shp
│   ├── travel/          travel_survey.csv
│   └── boundary/        boundary.shp
├── intermediate/
│   ├── 01_overlay/      road segments + avg_flood_depth
│   ├── 02_status/       road segments + status classification
│   ├── 03_if/           if.csv
│   ├── 04_accessibility/ accessibility.csv
│   ├── 05_behaviour/    behaviour.csv
│   ├── 06_ba/           ba.csv
│   ├── 07_master/       master_resilience.csv
│   └── 08_tri/          tri.csv
├── output/
│   ├── maps/            PNG visualizations
│   ├── csv/             final CSV exports
│   └── shapefiles/      final shapefile exports
├── dummy_data/          copies of raw synthetic inputs (for inspection)
└── logs/                pipeline run logs
```


## Data Dictionary

### 1. Road Network (`input/roads/roads.shp`) — LineString

| Field | Type | Description | Units | Example |
|---|---|---|---|---|
| road_id | string | Unique road segment identifier | — | R001 |
| road_type | string | Functional road class | — | primary / secondary / residential |
| length_m | float | Segment length | metres | 1500.0 |
| speed_lim* | int | Posted speed limit | km/h | 60 |
| lanes | int | Number of lanes | count | 2 |
| status | string | Filled in later (Operational/Partial/Closed) | — | Operational |

\* stored as `speed_lim` in the shapefile because ESRI Shapefile field names are limited to 10
characters; conceptually this is `speed_limit`.

### 2. Flood Raster (`input/flood/flood_depth.tif`) — GeoTIFF, Float32, 10 m resolution

| Attribute | Description | Units |
|---|---|---|
| Pixel value | Flood inundation depth during the historical event | metres |
| Resolution | 10 × 10 | metres/pixel |
| nodata | -9999 | — |

### 3. Administrative Boundary (`input/boundary/boundary.shp`) — Polygon

| Field | Type | Description | Example |
|---|---|---|---|
| ward_id | string | Unique ward identifier | W01 |
| ward_name | string | Ward display name | Ward 1 |

### 4. Essential Facilities (`input/facilities/facilities.shp`) — Point

| Field | Type | Description | Example |
|---|---|---|---|
| fac_id* | string | Unique facility identifier | F001 |
| fac_type* | string | Facility category | Hospital / School / Market / Bus Stop / Metro Station |

\* stored as `fac_id` / `fac_type` (10-character shapefile field limit); conceptually
`facility_id` / `facility_type`.

### 5. Travel Survey (`input/travel/travel_survey.csv`) — CSV

| Field | Type | Description | Units | Example |
|---|---|---|---|---|
| trip_id | string | Unique trip identifier | — | T0001 |
| person_id | string | Unique respondent identifier | — | P0042 |
| origin_x, origin_y | float | Trip origin coordinates | metres (projected CRS) | 1234.5, 987.6 |
| destination_x, destination_y | float | Trip destination coordinates | metres (projected CRS) | 2200.0, 1500.0 |
| planned_mode | string | Mode the traveller intended to use | — | car |
| actual_mode | string | Mode actually used | — | bus |
| travel_time_before | float | Typical (pre-event) travel time | minutes | 25.0 |
| travel_time_after | float | Observed travel time during/after the event | minutes | 48.0 |
| trip_cancelled | bool | Whether the trip was abandoned | — | False |
| route_changed | bool | Whether the traveller rerouted | — | True |
| ward_id | string | Ward the trip originates in | — | W03 |

All coordinates use the same projected CRS as the spatial layers (**EPSG:32646**) so that roads,
boundary, facilities, flood raster, and travel origins/destinations are all spatially consistent
and directly overlay one another.


## Cell: Install required packages

Colab ships with some geospatial packages but not all of them at compatible versions, so we pin
installs explicitly. This cell is safe to re-run.

In [ ]:
# Install required packages (Colab-safe, idempotent)
!pip install -q geopandas rasterio rasterstats networkx shapely scipy matplotlib
print("Packages installed.")

## Cell: Import libraries

In [ ]:
from __future__ import annotations

import os
import logging
import warnings
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.transform import from_origin
import networkx as nx
from shapely.geometry import LineString, Point, Polygon, box
from scipy import stats
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

print("Libraries imported successfully.")

## Cell: Create project folder structure

In [ ]:
# ---------------------------------------------------------------------------
# Function: create_project_structure
# Purpose : Create the full directory tree used by the entire pipeline.
# Inputs  : base_dir (str) - root folder for the project
# Output  : Nested folders on disk (no files)
# Returns : dict[str, Path] mapping a short key to every created folder
# ---------------------------------------------------------------------------
def create_project_structure(base_dir: str = "/content/project") -> dict:
    """Create the standard TRI project folder structure.

    Parameters
    ----------
    base_dir : str
        Root directory under which the project tree is created.

    Returns
    -------
    dict[str, Path]
        Mapping of short keys (e.g. "roads", "if", "maps") to their
        absolute Path objects, for convenient reuse in later cells.
    """
    base = Path(base_dir)
    subfolders = {
        "roads": "input/roads",
        "flood": "input/flood",
        "facilities": "input/facilities",
        "travel": "input/travel",
        "boundary": "input/boundary",
        "overlay": "intermediate/01_overlay",
        "status": "intermediate/02_status",
        "if": "intermediate/03_if",
        "accessibility": "intermediate/04_accessibility",
        "behaviour": "intermediate/05_behaviour",
        "ba": "intermediate/06_ba",
        "master": "intermediate/07_master",
        "tri": "intermediate/08_tri",
        "maps": "output/maps",
        "csv": "output/csv",
        "shapefiles": "output/shapefiles",
        "dummy_data": "dummy_data",
        "logs": "logs",
    }
    paths = {}
    for key, rel in subfolders.items():
        p = base / rel
        p.mkdir(parents=True, exist_ok=True)
        paths[key] = p
    paths["base"] = base
    return paths


PROJECT = create_project_structure("/content/project")

# ---------------------------------------------------------------------------
# Logging setup: every function in this pipeline logs to logs/pipeline.log
# ---------------------------------------------------------------------------
LOG_PATH = PROJECT["logs"] / "pipeline.log"
logger = logging.getLogger("TRI")
logger.setLevel(logging.INFO)
logger.handlers.clear()
fh = logging.FileHandler(LOG_PATH)
fh.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
logger.addHandler(fh)
sh = logging.StreamHandler()
sh.setFormatter(logging.Formatter("%(levelname)s | %(message)s"))
logger.addHandler(sh)

logger.info("Project structure created at %s", PROJECT["base"])
for k, v in PROJECT.items():
    print(f"{k:12s} -> {v}")

## Cell: Validation helper

A single reusable helper is used after **every** function in this pipeline (per the project
specification) to print input/output row counts, missing values, summary statistics, and the
file location that was written.

In [ ]:
# ---------------------------------------------------------------------------
# Function: validate_output
# Purpose : Standard post-processing validation report used after every
#           pipeline function (vector, raster, or tabular output).
# Inputs  : label (str), df_in (optional DataFrame/GeoDataFrame),
#           df_out (optional DataFrame/GeoDataFrame), file_path (str)
# Output  : Printed report (no file written)
# Returns : None
# ---------------------------------------------------------------------------
def validate_output(label: str, df_out, file_path: str, df_in=None) -> None:
    """Print a standard validation report for a pipeline step.

    Parameters
    ----------
    label : str
        Human-readable name of the processing step (e.g. "Road-Flood Overlay").
    df_out : DataFrame or GeoDataFrame
        The output dataset produced by the step.
    file_path : str
        Path to the file that was written to disk.
    df_in : DataFrame or GeoDataFrame, optional
        The input dataset, if row-count comparison is meaningful.
    """
    print(f"\n--- Validation: {label} ---")
    if df_in is not None:
        print(f"Input rows : {len(df_in)}")
    print(f"Output rows: {len(df_out)}")

    numeric = df_out.select_dtypes(include=[np.number])
    if not numeric.empty:
        missing = int(numeric.isna().sum().sum())
        print(f"Missing values (numeric cols): {missing}")
        print("Summary statistics:")
        print(numeric.describe().T[["mean", "std", "min", "max"]])
    else:
        print("Missing values: n/a (no numeric columns)")

    print(f"File written to: {file_path}")
    logger.info("%s -> %s (rows=%d)", label, file_path, len(df_out))

## Cell: Dummy data generation — study area and administrative boundary

All synthetic datasets share the projected coordinate system **EPSG:32646** and are built from
the *same* geometric skeleton (a 5 km × 5 km study area split into a 3×3 ward grid), which is
what guarantees that roads intersect the flood raster, facilities sit inside the boundary,
and travel origins/destinations sit on the road network — everything is generated from one
spatially consistent source of truth rather than independently.

In [ ]:
# ---------------------------------------------------------------------------
# Function: generate_boundary
# Purpose : Create a synthetic administrative boundary made of 9 wards
#           (3x3 grid) covering the study area.
# Inputs  : output_shp (str), extent (tuple), n_x (int), n_y (int)
# Output  : Polygon shapefile with fields [ward_id, ward_name]
# Returns : (output_shp path, GeoDataFrame)
# ---------------------------------------------------------------------------
CRS = "EPSG:32646"
STUDY_EXTENT = (0, 0, 5000, 5000)  # minx, miny, maxx, maxy (metres)


def generate_boundary(
    output_shp: str,
    extent: tuple = STUDY_EXTENT,
    n_x: int = 3,
    n_y: int = 3,
    seed: int = 42,
) -> tuple:
    """Generate a synthetic ward-boundary polygon shapefile.

    Parameters
    ----------
    output_shp : str
        Destination path for the boundary shapefile.
    extent : tuple
        (minx, miny, maxx, maxy) of the study area in metres.
    n_x, n_y : int
        Number of wards along each axis (n_x * n_y wards total).
    seed : int
        Random seed for reproducibility.

    Returns
    -------
    tuple(str, geopandas.GeoDataFrame)
        Path to the written shapefile and the GeoDataFrame itself.
    """
    np.random.seed(seed)
    minx, miny, maxx, maxy = extent
    dx, dy = (maxx - minx) / n_x, (maxy - miny) / n_y

    polys, ward_ids, ward_names = [], [], []
    wid = 1
    for i in range(n_x):
        for j in range(n_y):
            polys.append(box(minx + i * dx, miny + j * dy, minx + (i + 1) * dx, miny + (j + 1) * dy))
            ward_ids.append(f"W{wid:02d}")
            ward_names.append(f"Ward {wid}")
            wid += 1

    gdf = gpd.GeoDataFrame({"ward_id": ward_ids, "ward_name": ward_names}, geometry=polys, crs=CRS)
    Path(output_shp).parent.mkdir(parents=True, exist_ok=True)
    gdf.to_file(output_shp)
    return output_shp, gdf


boundary_path, boundary_gdf = generate_boundary(str(PROJECT["boundary"] / "boundary.shp"))
validate_output("Generate Boundary", boundary_gdf, boundary_path)

## Cell: Dummy data generation — road network

In [ ]:
# ---------------------------------------------------------------------------
# Function: generate_roads
# Purpose : Create a synthetic road network (grid + residential connectors)
#           that spans the full study extent, guaranteeing it intersects
#           both the boundary and the flood raster generated later.
# Inputs  : output_shp (str), boundary_gdf (GeoDataFrame), extent (tuple)
# Output  : LineString shapefile, fields [road_id, road_type, length_m,
#           speed_lim, lanes, status]
# Returns : (output_shp path, GeoDataFrame)
# ---------------------------------------------------------------------------
def generate_roads(
    output_shp: str,
    extent: tuple = STUDY_EXTENT,
    n_grid_x: int = 3,
    n_grid_y: int = 3,
    n_connectors: int = 12,
    seed: int = 42,
) -> tuple:
    """Generate a synthetic road network covering the study extent.

    Parameters
    ----------
    output_shp : str
        Destination path for the road shapefile.
    extent : tuple
        (minx, miny, maxx, maxy) of the study area in metres.
    n_grid_x, n_grid_y : int
        Number of grid lines along each axis (aligned with ward boundaries).
    n_connectors : int
        Number of extra residential connector segments to add for density.
    seed : int
        Random seed for reproducibility.

    Returns
    -------
    tuple(str, geopandas.GeoDataFrame)
    """
    np.random.seed(seed)
    minx, miny, maxx, maxy = extent
    dx, dy = (maxx - minx) / n_grid_x, (maxy - miny) / n_grid_y
    road_types = ["primary", "secondary", "residential"]

    records = []
    rid = 1
    for j in range(n_grid_y + 1):
        y = miny + j * dy
        records.append((f"R{rid:03d}", np.random.choice(road_types), LineString([(minx, y), (maxx, y)])))
        rid += 1
    for i in range(n_grid_x + 1):
        x = minx + i * dx
        records.append((f"R{rid:03d}", np.random.choice(road_types), LineString([(x, miny), (x, maxy)])))
        rid += 1
    for _ in range(n_connectors):
        x1, y1 = np.random.uniform(minx, maxx), np.random.uniform(miny, maxy)
        x2 = min(max(x1 + np.random.uniform(-500, 500), minx), maxx)
        y2 = min(max(y1 + np.random.uniform(-500, 500), miny), maxy)
        records.append((f"R{rid:03d}", "residential", LineString([(x1, y1), (x2, y2)])))
        rid += 1

    road_ids = [r[0] for r in records]
    r_types = [r[1] for r in records]
    geoms = [r[2] for r in records]
    lengths = [g.length for g in geoms]
    speed_lim = [int(np.random.choice([80, 60, 40, 30])) for _ in records]
    lanes = [int(np.random.choice([4, 2, 2, 1])) for _ in records]
    status = ["unknown"] * len(records)

    gdf = gpd.GeoDataFrame(
        {
            "road_id": road_ids,
            "road_type": r_types,
            "length_m": lengths,
            "speed_lim": speed_lim,
            "lanes": lanes,
            "status": status,
        },
        geometry=geoms,
        crs=CRS,
    )
    Path(output_shp).parent.mkdir(parents=True, exist_ok=True)
    gdf.to_file(output_shp)
    return output_shp, gdf


roads_path, roads_gdf = generate_roads(str(PROJECT["roads"] / "roads.shp"))
validate_output("Generate Roads", roads_gdf, roads_path)

## Cell: Dummy data generation — flood raster

In [ ]:
# ---------------------------------------------------------------------------
# Function: generate_flood_raster
# Purpose : Create a synthetic flood-depth GeoTIFF (10 m resolution) whose
#           footprint exactly matches the study extent, guaranteeing every
#           road segment overlaps flooded pixels somewhere along its length.
# Inputs  : output_tif (str), extent (tuple), resolution (int, metres)
# Output  : Float32 GeoTIFF, single band, flood depth in metres
# Returns : output_tif path
# ---------------------------------------------------------------------------
def generate_flood_raster(
    output_tif: str,
    extent: tuple = STUDY_EXTENT,
    resolution: int = 10,
    seed: int = 42,
) -> str:
    """Generate a synthetic flood-depth raster covering the study extent.

    A diagonal high-depth corridor (simulating an overflowing river) is
    combined with random noise to produce a spatially realistic depth
    surface, ranging roughly 0-1.5 m.

    Parameters
    ----------
    output_tif : str
        Destination path for the GeoTIFF.
    extent : tuple
        (minx, miny, maxx, maxy) of the study area in metres.
    resolution : int
        Pixel size in metres.
    seed : int
        Random seed for reproducibility.

    Returns
    -------
    str
        Path to the written raster.
    """
    np.random.seed(seed)
    minx, miny, maxx, maxy = extent
    width = int((maxx - minx) / resolution)
    height = int((maxy - miny) / resolution)
    transform = from_origin(minx, maxy, resolution, resolution)

    xs = np.linspace(minx, maxx, width)
    ys = np.linspace(maxy, miny, height)
    xx, yy = np.meshgrid(xs, ys)

    dist_to_river = np.abs(xx - yy)  # proxy distance from the diagonal "river"
    depth = np.clip(1.2 - dist_to_river / 2000, 0, None) + np.random.rand(height, width) * 0.15
    depth = depth.astype("float32")

    Path(output_tif).parent.mkdir(parents=True, exist_ok=True)
    with rasterio.open(
        output_tif, "w", driver="GTiff", height=height, width=width,
        count=1, dtype="float32", crs=CRS, transform=transform, nodata=-9999,
    ) as dst:
        dst.write(depth, 1)

    return output_tif


flood_path = generate_flood_raster(str(PROJECT["flood"] / "flood_depth.tif"))

with rasterio.open(flood_path) as src:
    arr = src.read(1)
print("\n--- Validation: Generate Flood Raster ---")
print(f"Raster shape : {arr.shape}")
print(f"Depth range  : {arr.min():.2f} m to {arr.max():.2f} m")
print(f"Mean depth   : {arr.mean():.2f} m")
print(f"File written to: {flood_path}")
logger.info("Generate Flood Raster -> %s (shape=%s)", flood_path, arr.shape)

## Cell: Dummy data generation — essential facilities

In [ ]:
# ---------------------------------------------------------------------------
# Function: generate_facilities
# Purpose : Create synthetic essential-facility points, constrained to lie
#           strictly inside the study area (and therefore inside the union
#           of ward polygons).
# Inputs  : output_shp (str), boundary_gdf (GeoDataFrame), n_facilities (int)
# Output  : Point shapefile, fields [fac_id, fac_type]
# Returns : (output_shp path, GeoDataFrame)
# ---------------------------------------------------------------------------
def generate_facilities(
    output_shp: str,
    extent: tuple = STUDY_EXTENT,
    n_facilities: int = 25,
    seed: int = 42,
) -> tuple:
    """Generate synthetic essential-facility points inside the study area.

    Parameters
    ----------
    output_shp : str
        Destination path for the facilities shapefile.
    extent : tuple
        (minx, miny, maxx, maxy) of the study area in metres.
    n_facilities : int
        Number of facility points to generate.
    seed : int
        Random seed for reproducibility.

    Returns
    -------
    tuple(str, geopandas.GeoDataFrame)
    """
    np.random.seed(seed)
    minx, miny, maxx, maxy = extent
    fac_types = ["Hospital", "School", "Market", "Bus Stop", "Metro Station"]

    fx = np.random.uniform(minx + 50, maxx - 50, n_facilities)
    fy = np.random.uniform(miny + 50, maxy - 50, n_facilities)

    gdf = gpd.GeoDataFrame(
        {
            "fac_id": [f"F{i+1:03d}" for i in range(n_facilities)],
            "fac_type": [np.random.choice(fac_types) for _ in range(n_facilities)],
        },
        geometry=[Point(x, y) for x, y in zip(fx, fy)],
        crs=CRS,
    )
    Path(output_shp).parent.mkdir(parents=True, exist_ok=True)
    gdf.to_file(output_shp)
    return output_shp, gdf


facilities_path, facilities_gdf = generate_facilities(str(PROJECT["facilities"] / "facilities.shp"))
validate_output("Generate Facilities", facilities_gdf, facilities_path)

## Cell: Dummy data generation — travel survey

Trip origins and destinations are sampled from the **actual vertices of the road network**
generated above (with small positional jitter), so every trip starts and ends on or near a real
road segment, and each trip's `ward_id` is assigned by a true point-in-polygon test against the
boundary layer.

In [ ]:
# ---------------------------------------------------------------------------
# Function: generate_travel_survey
# Purpose : Create a synthetic household travel survey whose origins and
#           destinations are drawn from real road-network vertices, and
#           whose ward_id is assigned via point-in-polygon against the
#           boundary layer, guaranteeing spatial consistency with the
#           other datasets.
# Inputs  : output_csv (str), roads_gdf (GeoDataFrame), boundary_gdf (GeoDataFrame)
# Output  : CSV, fields per the project data dictionary
# Returns : output_csv path
# ---------------------------------------------------------------------------
def generate_travel_survey(
    output_csv: str,
    roads_gdf: gpd.GeoDataFrame,
    boundary_gdf: gpd.GeoDataFrame,
    n_trips: int = 200,
    seed: int = 42,
) -> str:
    """Generate a synthetic travel survey consistent with the road network.

    Parameters
    ----------
    output_csv : str
        Destination path for the travel-survey CSV.
    roads_gdf : geopandas.GeoDataFrame
        The road network used to source realistic origin/destination points.
    boundary_gdf : geopandas.GeoDataFrame
        Ward polygons, used to assign each trip's ward_id.
    n_trips : int
        Number of synthetic trips to generate.
    seed : int
        Random seed for reproducibility.

    Returns
    -------
    str
        Path to the written CSV.
    """
    np.random.seed(seed)
    all_coords = np.array([pt for geom in roads_gdf.geometry for pt in geom.coords])

    modes = ["car", "bus", "walk", "rickshaw", "motorbike"]
    idx_o = np.random.choice(len(all_coords), n_trips)
    idx_d = np.random.choice(len(all_coords), n_trips)

    rows = []
    for t in range(n_trips):
        ox, oy = all_coords[idx_o[t]] + np.random.normal(0, 20, 2)
        dx_, dy_ = all_coords[idx_d[t]] + np.random.normal(0, 20, 2)
        planned = np.random.choice(modes)
        actual = planned if np.random.rand() > 0.25 else np.random.choice(modes)
        tt_before = np.random.uniform(5, 60)
        tt_after = tt_before * np.random.uniform(1.0, 2.5)
        cancelled = bool(np.random.rand() < 0.1)
        route_changed = bool(np.random.rand() < 0.3)

        pt = Point(ox, oy)
        match = boundary_gdf[boundary_gdf.contains(pt)]
        ward_id = match.iloc[0]["ward_id"] if len(match) > 0 else np.random.choice(boundary_gdf["ward_id"])

        rows.append([
            f"T{t+1:04d}", f"P{np.random.randint(1, 150):04d}", ox, oy, dx_, dy_,
            planned, actual, round(tt_before, 1), round(tt_after, 1),
            cancelled, route_changed, ward_id,
        ])

    df = pd.DataFrame(rows, columns=[
        "trip_id", "person_id", "origin_x", "origin_y", "destination_x", "destination_y",
        "planned_mode", "actual_mode", "travel_time_before", "travel_time_after",
        "trip_cancelled", "route_changed", "ward_id",
    ])
    Path(output_csv).parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(output_csv, index=False)
    return output_csv


travel_path = generate_travel_survey(
    str(PROJECT["travel"] / "travel_survey.csv"), roads_gdf, boundary_gdf
)
travel_df = pd.read_csv(travel_path)
validate_output("Generate Travel Survey", travel_df, travel_path)

## Cell: Part 1 summary

All five synthetic datasets have now been generated and written to disk under
`/content/project/input/`, and are spatially consistent with one another (shared CRS,
shared study extent, roads sourced from the same skeleton as trip origins/destinations,
facilities and boundary sharing the same footprint).

**Files on disk:**
- `input/boundary/boundary.shp`
- `input/roads/roads.shp`
- `input/flood/flood_depth.tif`
- `input/facilities/facilities.shp`
- `input/travel/travel_survey.csv`

**Reusable objects still in memory for convenience** (though every downstream function in Parts
2-4 will read from disk, not from these variables, per the project's file-based-pipeline
requirement): `PROJECT`, `boundary_gdf`, `roads_gdf`, `facilities_gdf`, `travel_df`, `logger`,
`validate_output()`.

**Next: Part 2** — Road-Flood Overlay → Road Status Classification → Infrastructure
Functionality (IF) → Accessibility Index (AI).
